# Father — post-mortem investigation

**Premise.** Monitoring reported an unexplained `sshd` restart and an unrecognised session on this
host. The system was powered off; disk and memory were acquired. Nothing about the cause is assumed
here — the mechanism has to come out of the evidence.

**Scope.** The preserved disk and memory acquisitions of one run, plus the prepared extractions
derived from them.

**Boundary.** Scenario execution records guided the design of this lab and are disclosed in
Section 7. They are not evidence: every finding must cite an acquired disk, timeline or RAM record.

In [ ]:
import json, os, re, sys
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd

RUN_ID = "father-u22-20260913-01"

PROJECT_ROOT = next(
    p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / "shared" / "experiments").is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT))
from investigations.common import forensics as fx

case = fx.load_case(PROJECT_ROOT, RUN_ID)
prepared = json.loads((case.prepared / "prepare.json").read_text())

# Work from the case directory so every command below reads as a short, quotable path.
os.chdir(case.run_root)
IMG = str(case.disk_image.relative_to(case.run_root))
OUT, DATA = Path("investigation/output"), Path("investigation/data")
for d in (OUT, DATA):
    d.mkdir(parents=True, exist_ok=True)


def product(name):
    "Path of a prepared product, echoing the invocation that produced it."
    p = prepared["products"][name]
    root = str(case.run_root) + "/"
    print("$ " + " ".join(a.replace(root, "") for a in p["argv"]))
    print(f"# recorded: {p['state']}, exit {p.get('exit_code')}")
    return Path(p["path"])


def istat_times(text):
    "The four inode times from istat output, as UTC timestamps."
    labels = {"atime": "Accessed", "mtime": "File Modified",
              "ctime": "Inode Modified", "crtime": "File Created"}
    out = {}
    for key, label in labels.items():
        m = re.search(rf"^{label}:\s*(.+?)\s*$", text, re.M)
        out[key] = pd.to_datetime(re.sub(r"\s+\([A-Z]+\)$", "", m.group(1)), utc=True) if m else pd.NaT
    return out


SECTOR_SIZE = int(re.search(r"Units are in (\d+)-byte sectors",
                            Path(prepared["products"]["mmls"]["path"]).read_text()).group(1))
ROOT = [(int(p["argv"][p["argv"].index("-o") + 1]), name)
        for name, p in prepared["products"].items()
        if name.startswith("fsstat-") and p["state"] == "ok"
        and "File System Type: Ext4" in Path(p["path"]).read_text()]
print(f"Ext4 candidates: {ROOT}")
ROOT_OFFSET, ROOT_FS_PRODUCT = ROOT[0]

## 0. Evidence and scope

### 0.1 What evidence is this, and when was it acquired?

In [16]:
print(f"{case.manifest['scenario']} on {case.platform['guest_os']}, "
      f"kernel {case.platform['kernel']} ({case.platform['arch']}), "
      f"recorded timezone {case.timezone}")

fx.show(pd.DataFrame([
    {"source": src, "path": rec["path"], "bytes": rec["size_bytes"],
     "recorded_sha256": rec["sha256"], "started_at": rec["started_at"],
     "ended_at": rec["ended_at"]}
    for src, rec in (("disk", case.acquisition["disk"]), ("ram", case.acquisition["memory"]))
]), n=2, caption="Acquired evidence")

father on Ubuntu 22.04.5 LTS, kernel 5.15.0-179-generic (x86_64), recorded timezone Etc/UTC


**Acquired evidence — showing 2 of 2 rows**

,source,path,bytes,recorded_sha256,started_at,ended_at
0,disk,disk/evidence_disk.E01,10737418240,ae591cfdfb26569b16478bbbc80bdd4e7f9bcc1741a84b...,2026-09-13T20:46:38.398053Z,2026-09-13T20:47:16.034097Z
1,ram,memory/mem.raw,2147747795,e4089156b81a250dda36391932bf8011f70d0b54cd7acd...,2026-09-13T20:46:30.996505Z,2026-09-13T20:46:32.853530Z


### 0.2 Is the evidence internally consistent?

The EWF verification ran once during preparation and is not repeated here.

In [ ]:
fx.sh(f"grep -E 'hash calculated|SUCCESS|FAILURE' {product('ewfverify')}",
      label="s0-02-ewfverify", out_dir=OUT)

### 0.3 What limits this examination?

RAM and disk were acquired at different moments, so they are two states, not one snapshot. The
guest runs a vanilla logging profile, so absent records are bounded negatives rather than evidence
of absence.

In [18]:
RAM_ENDED = pd.to_datetime(case.acquisition["memory"]["ended_at"], utc=True)
DISK_STARTED = pd.to_datetime(case.acquisition["disk"]["started_at"], utc=True)
print(f"RAM capture ended  {RAM_ENDED.isoformat()}")
print(f"disk image started {DISK_STARTED.isoformat()}")
print(f"gap {(DISK_STARTED - RAM_ENDED).total_seconds():.3f} s — anything the guest wrote in that "
      f"window is on disk but not in memory")

RAM capture ended  2026-09-13T20:46:32.853530+00:00
disk image started 2026-09-13T20:46:38.398053+00:00
gap 5.545 s — anything the guest wrote in that window is on disk but not in memory


### 0.4 How should timestamps be read?

All output below is displayed in UTC.

In [19]:
tz_inode, _ = fx.resolve(IMG, ROOT_OFFSET, "/etc/timezone")
fx.sh(f"icat -o {ROOT_OFFSET} {IMG} {tz_inode}", label="s0-04-timezone", out_dir=OUT)

lt_inode, lt_chain = fx.resolve(IMG, ROOT_OFFSET, "/etc/localtime")
fx.sh(f"istat -o {ROOT_OFFSET} -z UTC {IMG} {lt_inode}",
      label="s0-04-localtime-istat", out_dir=OUT, tail=12)
print(f"/etc/localtime -> {lt_chain[-1]['path']}")
print(f"examiner timezone: {datetime.now().astimezone().tzinfo}")

$ ifind -o 227328 -n /etc dumps/disk/evidence_disk.E01
$ istat -o 227328 dumps/disk/evidence_disk.E01 41
$ ifind -o 227328 -n /etc/timezone dumps/disk/evidence_disk.E01
$ istat -o 227328 dumps/disk/evidence_disk.E01 1290
$ icat -o 227328 dumps/disk/evidence_disk.E01 1290
Etc/UTC
$ ifind -o 227328 -n /etc dumps/disk/evidence_disk.E01
$ istat -o 227328 dumps/disk/evidence_disk.E01 41
$ ifind -o 227328 -n /etc/localtime dumps/disk/evidence_disk.E01
$ istat -o 227328 dumps/disk/evidence_disk.E01 651
$ ifind -o 227328 -n /usr dumps/disk/evidence_disk.E01
$ istat -o 227328 dumps/disk/evidence_disk.E01 1582
$ ifind -o 227328 -n /usr/share dumps/disk/evidence_disk.E01
$ istat -o 227328 dumps/disk/evidence_disk.E01 17230
$ ifind -o 227328 -n /usr/share/zoneinfo dumps/disk/evidence_disk.E01
$ istat -o 227328 dumps/disk/evidence_disk.E01 33460
$ ifind -o 227328 -n /usr/share/zoneinfo/Etc dumps/disk/evidence_disk.E01
$ istat -o 227328 dumps/disk/evidence_disk.E01 33874
$ ifind -o 227328 -n /usr/sh

**Interpretation.** _(to write)_

## 1. Disk: what happened on this filesystem, and what persists?

### 1.1 Which partition holds the root filesystem?

In [30]:
fx.sh(f"cat {product('mmls')}", label="s1-01-mmls", out_dir=OUT)
fx.sh(f"head -n 30 {product(ROOT_FS_PRODUCT)}", label="s1-01-fsstat", out_dir=OUT)
print(f"root filesystem at sector {ROOT_OFFSET} ({SECTOR_SIZE}-byte sectors)")

$ /usr/local/bin/mmls /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/dumps/disk/evidence_disk.E01
# recorded: ok, exit 0
$ cat investigation/prepared/raw/mmls.txt
GUID Partition Table (EFI)
Offset Sector: 0
Units are in 512-byte sectors

      Slot      Start        End          Length       Description
000:  Meta      0000000000   0000000000   0000000001   Safety Table
001:  -------   0000000000   0000002047   0000002048   Unallocated
002:  Meta      0000000001   0000000001   0000000001   GPT Header
003:  Meta      0000000002   0000000033   0000000032   Partition Table
004:  013       0000002048   0000010239   0000008192   
005:  014       0000010240   0000227327   0000217088   
006:  000       0000227328   0020971486   0020744159   
007:  -------   0020971487   0020971519   0000000033   Unallocated
$ /usr/local/bin/fsstat -o 0000227328 /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/dumps/disk/evidence_disk.E01
# record

### 1.2 What changed on this filesystem shortly before acquisition?

No mechanism is assumed. The bodyfile is queried for every allocated object whose mtime, ctime or
crtime falls in the 24 hours before the disk was imaged, ordered by time. The window is a stated
choice, bounded by the acquisition time; a longer window is one edit away.

In [27]:
BODY = fx.load_bodyfile(product("allocated.body"))
ALLOCATED = BODY[BODY["name_state"].eq("allocated")]

WINDOW_START = DISK_STARTED - timedelta(hours=24)
in_window = ALLOCATED[
    ALLOCATED[["mtime_utc", "ctime_utc", "crtime_utc"]].ge(WINDOW_START).any(axis=1)
].copy()
in_window["last_change_utc"] = in_window[["mtime_utc", "ctime_utc", "crtime_utc"]].max(axis=1)
in_window = in_window.sort_values("last_change_utc")

in_window.to_csv(OUT / "s1-02-recent-changes.csv", index=False)
fx.show(in_window, cols=("last_change_utc", "name", "inode", "size", "mtime_utc",
                         "ctime_utc", "crtime_utc", "locator"),
        n=60, caption=f"Allocated objects changed after {WINDOW_START.isoformat()}")
print(f"{len(in_window)} objects in window; full result: {OUT}/s1-02-recent-changes.csv")

$ /usr/local/bin/fls -r -m / -o 227328 /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/dumps/disk/evidence_disk.E01
# recorded: ok, exit 0


**Allocated objects changed after 2026-09-12T20:46:38.398053+00:00 — showing 60 of 82 rows**

,last_change_utc,name,inode,size,mtime_utc,ctime_utc,crtime_utc,locator
2192,2026-09-13 20:44:49+00:00,/tmp/.font-unix,258071,4096,2026-09-13 20:44:49+00:00,2026-09-13 20:44:49+00:00,2026-09-13 20:44:49+00:00,allocated.body#L2193
74476,2026-09-13 20:44:49+00:00,/var/log/journal/5197e2566dcb4cd799bf2ba16efa2...,74301,8388608,2026-09-13 20:44:49+00:00,2026-09-13 20:44:49+00:00,2026-08-11 17:13:54+00:00,allocated.body#L74477
74475,2026-09-13 20:44:49+00:00,/var/log/journal/5197e2566dcb4cd799bf2ba16efa2...,74157,8388608,2026-09-13 20:44:49+00:00,2026-09-13 20:44:49+00:00,2026-08-11 17:13:43+00:00,allocated.body#L74476
2189,2026-09-13 20:44:49+00:00,/tmp/.X11-unix,258068,4096,2026-09-13 20:44:49+00:00,2026-09-13 20:44:49+00:00,2026-09-13 20:44:49+00:00,allocated.body#L2190
2190,2026-09-13 20:44:49+00:00,/tmp/.ICE-unix,258069,4096,2026-09-13 20:44:49+00:00,2026-09-13 20:44:49+00:00,2026-09-13 20:44:49+00:00,allocated.body#L2191
2191,2026-09-13 20:44:49+00:00,/tmp/.XIM-unix,258070,4096,2026-09-13 20:44:49+00:00,2026-09-13 20:44:49+00:00,2026-09-13 20:44:49+00:00,allocated.body#L2192
2193,2026-09-13 20:44:49+00:00,/tmp/.Test-unix,258072,4096,2026-09-13 20:44:49+00:00,2026-09-13 20:44:49+00:00,2026-09-13 20:44:49+00:00,allocated.body#L2194
74230,2026-09-13 20:44:50+00:00,/var/lib/cloud/instance -> /var/lib/cloud/inst...,74165,41,2026-09-13 20:44:50+00:00,2026-09-13 20:44:50+00:00,2026-09-13 20:44:50+00:00,allocated.body#L74231
74194,2026-09-13 20:44:53+00:00,/var/lib/cloud/instances/lab-ubuntu-22.04/scripts,74213,4096,2026-08-11 17:13:45+00:00,2026-09-13 20:44:53+00:00,2026-08-11 17:13:45+00:00,allocated.body#L74195
74193,2026-09-13 20:44:53+00:00,/var/lib/cloud/instances/lab-ubuntu-22.04/hand...,74212,4096,2026-08-11 17:13:45+00:00,2026-09-13 20:44:53+00:00,2026-08-11 17:13:45+00:00,allocated.body#L74194


82 objects in window; full result: investigation/output/s1-02-recent-changes.csv


**Interpretation.** _(to write — which of these are ordinary boot/shutdown activity, and which are
not? note anything whose timestamps sit near or after the acquisition times)_

### 1.3 Which persistence-relevant paths exist, and do any of them appear above?

Scope of this sweep, stated so the negative is bounded: the dynamic-loader preload file and
configuration directory, cron, systemd system units, rc links, profile scripts, and per-user shell
startup files. Anything outside this list was not examined here.

In [ ]:
LOCATIONS = [
    ("dynamic loader preload",  r"^/etc/ld\.so\.preload$"),
    ("dynamic loader config",   r"^/etc/ld\.so\.conf\.d(?:/|$)"),
    ("cron",                    r"^/etc/(?:crontab$|cron(?:\.|/|$))"),
    ("systemd system units",    r"^/etc/systemd/system(?:/|$)"),
    ("rc links",                r"^/etc/rc[0-6S]\.d(?:/|$)"),
    ("profile scripts",         r"^/etc/profile\.d(?:/|$)"),
    ("shell startup",           r"^/(?:root|home/[^/]+)/\.(?:bashrc|profile)$"),
]
hits = pd.concat(
    [ALLOCATED[ALLOCATED["name"].str.match(p, na=False)].assign(location=name)
     for name, p in LOCATIONS],
    ignore_index=True,
)
hits["in_window"] = hits["locator"].isin(in_window["locator"])
hits.to_csv(OUT / "s1-03-persistence-paths.csv", index=False)

print(hits.groupby("location").agg(entries=("name", "size"),
                                   changed_in_window=("in_window", "sum")).to_string())
print(f"\nfull inventory: {OUT}/s1-03-persistence-paths.csv\n")

fx.show(hits[hits["in_window"]].sort_values("mtime_utc", ascending=False),
        cols=("location", "name", "inode", "mode", "size", "mtime_utc"),
        n=20, caption="Persistence-location entries that changed inside the window")

preload = hits[hits["location"].eq("dynamic loader preload")]
print(f"preload entries found: {len(preload)}")
PRELOAD_ROW = preload.iloc[0]
PRELOAD_PATH = str(PRELOAD_ROW["name"])

**Interpretation.** _(to write — a stock Ubuntu cloud image ships no `/etc/ld.so.preload`)_

### 1.4 What does that configuration file contain?

In [ ]:
preload_inode, _ = fx.resolve(IMG, ROOT_OFFSET, PRELOAD_PATH)
preload_istat = fx.sh(f"istat -o {ROOT_OFFSET} -z UTC {IMG} {preload_inode}",
                      label="s1-04-preload-istat", out_dir=OUT, tail=20)
preload_icat = fx.sh(f"icat -o {ROOT_OFFSET} {IMG} {preload_inode}",
                     label="s1-04-preload-icat", out_dir=OUT)

ENTRIES = [w for line in preload_icat.stdout.splitlines()
           for w in line.partition("#")[0].split()]
print(f"\nconfiguration entries: {ENTRIES}")
PRELOAD_OBJECT_PATH = ENTRIES[0]

**Interpretation.** _(to write)_

### 1.5 What is the referenced object?

In [ ]:
object_inode, object_chain = fx.resolve(IMG, ROOT_OFFSET, PRELOAD_OBJECT_PATH)
print(" -> ".join(f"{e['path']}({e['inode']})" + (f" [link {e['target']}]" if "target" in e else "")
                  for e in object_chain))

object_istat = fx.sh(f"istat -o {ROOT_OFFSET} -z UTC {IMG} {object_inode}",
                     label="s1-05-object-istat", out_dir=OUT, tail=20)

OBJECT_BIN = DATA / "s1-05-referenced-object.bin"
fx.sh(f"icat -o {ROOT_OFFSET} {IMG} {object_inode}",
      label=OBJECT_BIN.name, out_dir=DATA, binary=True, show=False)

fx.sh(f"file {OBJECT_BIN}", label="s1-05-object-file", out_dir=OUT)
fx.sh(f"sha256sum {OBJECT_BIN}", label="s1-05-object-sha256", out_dir=OUT)
fx.sh(f"readelf -h -d {OBJECT_BIN}", label="s1-05-object-readelf", out_dir=OUT, tail=20)
strings = fx.sh(f"strings -a -t x {OBJECT_BIN}", label="s1-05-object-strings",
                out_dir=OUT, tail=0)
print(f"\n{len(strings.stdout.splitlines())} strings preserved in {OUT}/s1-05-object-strings.txt")

## 2. Surrounding activity: who, when, and what else changed?

Driven by the strings and paths recovered in 1.5, plus the objects listed in 1.2: sessions and
logins, accounts and privilege, auth and service logs, shell history, and the `/tmp` and
`/dev/shm` inventories — enumerated before anything is selected.

**Not implemented.**

## 3. Memory: what was running?

Process view, which processes map the referenced object, environment, sockets and open files,
shell history from memory, recovery of the object from memory, and the kernel-mechanism checks
that should come back clean for a userland compromise.

**Not implemented.**

## 4. Deletion recovery

Deleted-entry enumeration, inode recovery, ext4 journal, then carving. Keep content, partial
content, metadata-only traces, bounded negatives and tool failures distinct.

**Not implemented.**

## 5. Chronology

`mactime` over the allocated bodyfile and the Plaso export, both scoped to the incident window,
merged into one sourced chronology. Acquisition and examiner artifacts get their own lane.

**Not implemented. Plaso required.**

## 6. Result tables

Findings are written here, by hand, from the observations above. Then the four tables and the
permitted counts, per [ai/RULES.md](../../ai/RULES.md).

**Not implemented.**

## 7. Validation against the controlled scenario

Compare the reconstruction with the run's command log: agreements, discrepancies, and what the
evidence could not show. Ground truth is an experimental reference, not a fourth source.

**Not implemented.**